In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

<font size="5" color="red">연관분석</font>
# 1. 연관분석 개요
- 데이터들 사이에 자주 발생하는 속성을 찾고, 그 속성들 사이에 연관성이 어느정도 있는지 분석
- 활동분야 : 이벤트 미리 감지, 신상품 카테고리 분석

# 2. 연관분석 구현

In [4]:
import csv
transaction = []
with open('data/cf_basket.csv', 'r', encoding='utf-8') as f :
    csvdata = csv.reader(f)
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [5]:
from apyori  import apriori
rules = apriori(transaction,
               min_support=0.15,
               min_confidence=0.1)
rules = list(rules)
len(rules)

18

In [6]:
rules[10]

RelationRecord(items=frozenset({'콜라', '소주'}), support=0.6, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'콜라', '소주'}), confidence=0.6, lift=1.0), OrderedStatistic(items_base=frozenset({'소주'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25), OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'소주'}), confidence=0.7499999999999999, lift=1.2499999999999998)])

In [16]:
rule = rules[10]
support = rule[1]
order_st = rule[2]
for item in order_st:
    lhs = item[0]
    rhs = item[1]
    confidence = item[2]
    lift = item[3]
    if lift > 1:
        print("{}=>{}\t {}\t {}\t {}".format(lhs, rhs, support, 
                                             round(confidence,2), 
                                             round(lift,2)))

frozenset({'소주'})=>frozenset({'콜라'})	 0.6	 1.0	 1.25
frozenset({'콜라'})=>frozenset({'소주'})	 0.6	 0.75	 1.25


In [17]:
for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = item[0]
        rhs = item[1]
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            print("{}=>{}\t {}\t {}\t {}".format(lhs, rhs, support, 
                                                 round(confidence,2), 
                                                 round(lift,2)))

frozenset({'맥주'})=>frozenset({'콜라'})	 0.4	 1.0	 1.25
frozenset({'콜라'})=>frozenset({'맥주'})	 0.4	 0.5	 1.25
frozenset({'소주'})=>frozenset({'콜라'})	 0.6	 1.0	 1.25
frozenset({'콜라'})=>frozenset({'소주'})	 0.6	 0.75	 1.25
frozenset({'콜라'})=>frozenset({'맥주', '소주'})	 0.2	 0.25	 1.25
frozenset({'맥주', '소주'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25
frozenset({'맥주'})=>frozenset({'콜라', '와인'})	 0.2	 0.5	 1.25
frozenset({'콜라'})=>frozenset({'맥주', '와인'})	 0.2	 0.25	 1.25
frozenset({'맥주', '와인'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25
frozenset({'콜라', '와인'})=>frozenset({'맥주'})	 0.2	 0.5	 1.25
frozenset({'소주'})=>frozenset({'콜라', '오렌지주스'})	 0.2	 0.33	 1.67
frozenset({'콜라'})=>frozenset({'오렌지주스', '소주'})	 0.2	 0.25	 1.25
frozenset({'오렌지주스', '소주'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25
frozenset({'콜라', '오렌지주스'})=>frozenset({'소주'})	 0.2	 1.0	 1.67
frozenset({'콜라'})=>frozenset({'와인', '소주'})	 0.2	 0.25	 1.25
frozenset({'와인', '소주'})=>frozenset({'콜라'})	 0.2	 1.0	 1.25


In [27]:
import pandas as pd

# 빈 데이터프레임 만들기
rules_df = pd.DataFrame(columns=['lhs', 'rhs', '지지도', '신뢰도', '향상도'])

idx = 0
for rule in rules:
    support = rule.support
    order_st = rule.ordered_statistics
    for item in order_st:
        lhs = ', '.join([data for data in item.items_base])   # 왼쪽 항목들
        rhs = ', '.join([data for data in item.items_add])    # 오른쪽 항목들
        confidence = item.confidence
        lift = item.lift
        if lift > 1:
            rules_df.loc[idx] = [lhs, rhs, support, round(confidence, 2), round(lift, 2)]
            idx += 1

rules_df.sort_values(by=['향상도','신뢰도'], ascending=False)


,lhs,rhs,지지도,신뢰도,향상도
13,"콜라, 오렌지주스",소주,0.2,1.00,1.67
10,소주,"콜라, 오렌지주스",0.2,0.33,1.67
0,맥주,콜라,0.4,1.00,1.25
2,소주,콜라,0.6,1.00,1.25
5,"맥주, 소주",콜라,0.2,1.00,1.25
8,"맥주, 와인",콜라,0.2,1.00,1.25
12,"오렌지주스, 소주",콜라,0.2,1.00,1.25
15,"와인, 소주",콜라,0.2,1.00,1.25
3,콜라,소주,0.6,0.75,1.25
1,콜라,맥주,0.4,0.50,1.25


# 3. 경주/전주 여행 자료 연관분석

In [42]:
import pandas as pd
from konlpy.tag import Hannanum
df = pd.read_csv('data/naver_kin.csv', sep='\t')
total_text_list = df['total_text'].to_list()
#total_text_list[:2]
analyzer = Hannanum()
total_noun_list = []
select_pos=['NC',"NQ"]  # 보통명사, 고유명사
불용어 = {'여행'}
for total_text in total_text_list:
   # total_noun = analyzer.nouns(total_text)
    total_noun = [token for token, tag in analyzer.pos(total_text, ntags=22)
                  if tag in select_pos and
                 token not in 불용어 and
                 len(token)>1]
    total_noun_list.append(total_noun)
print(total_noun_list)

[['전주', '가볼만한곳', '추천', '추억', '다양한체험', '7080감성', '추억여행', '테마박물관', '유익한시간', '전북투어패스', '통합이용권', '전북핫플', '여러여행지', '다양한체험', '카페이용추', '전주여행', '편안', '되시길', '감사'], ['전주여행', '전주여행', '아는사람', '호텔', '좋은가격', '2박3일여행인데', '얼마정도갖고가면좋을까요', '맛집같은거', '카페같은거', '추천', '안녕', '전주', '계획', '한옥마을', '근처'], ['전주', '관련', '질문', '전주', '계획하', '선택', '전주', '한옥마을', '자연경관', '음식', '가득', '동네', '시간', '거예요', '한옥마을', '근처', '장소'], ['중2', '여학생', '친구', '전주여행', '안녕', '여학생', '친한친구', '전주', '막막', '둘다', '애니', '좋아하는데전주', '오타쿠들'], ['부모닝', '전주여행', '부모님', '전주여행', '명소', '식당', '추천부탁드', '전주', '한옥마을', '유명', '완산구', '덕진구', '추억', '장소'], ['2박3', '전주여행', '이번주', '2박3', '여자친구', '전주여행', '여행경', '토요일', '실내데이트', '가능한곳', '감사', '숙소', '전주한옥마을', '근처', '한옥마을', '구경'], ['전주', '1박2', '저녁', '전주천', '산책길', '한옥마을', '근처', '야경', '일정', '관심', '테마', '카페', '체험', '추천', '전주', '되시길', '바랄게요'], ['전주', '코스', '추천이요~', '영화제', '기간', '일정', '2박3', '전주여행', '여행코스', '추천', '주목적', '전주국제영화제', '못할거에요', '전주하', '알짜배', '추천해주세욜', '전주'], ['전주', '전주', '커플여행', '추천', '숙소', '맛집', '추천', '감사', '한옥마을

In [44]:
%%time
rules = apriori(total_noun_list, min_support=0.15, min_confidence=0.3)
rules = list(rules)
len(rules)

CPU times: total: 1.11 s
Wall time: 1.11 s


644

In [45]:
rules_df.head(60)

,lhs,rhs,지지도,신뢰도,향상도
0,맥주,콜라,0.4,1.00,1.25
1,콜라,맥주,0.4,0.50,1.25
2,소주,콜라,0.6,1.00,1.25
3,콜라,소주,0.6,0.75,1.25
4,콜라,"맥주, 소주",0.2,0.25,1.25
5,"맥주, 소주",콜라,0.2,1.00,1.25
6,맥주,"콜라, 와인",0.2,0.50,1.25
7,콜라,"맥주, 와인",0.2,0.25,1.25
8,"맥주, 와인",콜라,0.2,1.00,1.25
9,"콜라, 와인",맥주,0.2,0.50,1.25
